## Notebook Guide

This notebook prepares the dataset for model training.

### Responsibilities

1. Build the image index.
2. Create the reproducible stratified 70/15/15 split.
3. Save train/validation/test metadata as CSV files.
4. Verify image preprocessing.
5. Build the TensorFlow input pipeline.
6. Verify batches before model training.

### Important

This notebook does NOT train the CNN or MobileNet models.

The output of this notebook is the verified dataset/input pipeline used by
the model notebooks.

# Fresh vs Rotten Fruit Classification — Preprocessing

This notebook prepares the fruit image dataset for model training.

## Preprocessing goals

1. Build a labeled image index.
2. Create the required 70% training, 15% validation, and 15% testing split.
3. Preserve class proportions using stratified splitting.
4. Convert images to RGB.
5. Resize images to a common input size.
6. Normalize pixel values before model training.

In [ ]:
# =========================================================
# RESUME / SETUP
# Run this cell first after every fresh Colab runtime.
# =========================================================

import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ---------------------------------------------------------
# 1. Clone / restore the GitHub repository
# ---------------------------------------------------------

REPO_URL = (
    "https://github.com/"
    "onkar-726/fresh-rotten-fruit-classification.git"
)

REPO_DIR = Path(
    "/content/fresh-rotten-fruit-classification"
)


if not REPO_DIR.exists():

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_DIR)
        ],
        check=True
    )

else:

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "pull",
            "--ff-only"
        ],
        check=True
    )


# Make the repository the working directory.
os.chdir(REPO_DIR)


print(
    "Repository:",
    REPO_DIR
)

print(
    "Working directory:",
    Path.cwd()
)


# ---------------------------------------------------------
# 2. Install project requirements
# ---------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "requirements.txt",
        "-q"
    ],
    check=True
)


print(
    "Project requirements installed."
)


# ---------------------------------------------------------
# 3. Add repository to Python path
# ---------------------------------------------------------

if str(REPO_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(REPO_DIR)
    )


# ---------------------------------------------------------
# 4. Download / restore Kaggle dataset
# ---------------------------------------------------------

import kagglehub


DATASET_HANDLE = (
    "sriramr/fruits-fresh-and-rotten-for-classification"
)


dataset_path = kagglehub.dataset_download(
    DATASET_HANDLE
)


DATASET_ROOT = (
    Path(dataset_path)
    / "dataset"
)


print(
    "Kaggle dataset path:",
    dataset_path
)

print(
    "Dataset root:",
    DATASET_ROOT
)


assert DATASET_ROOT.exists(), (
    f"Dataset root not found: {DATASET_ROOT}"
)


# ---------------------------------------------------------
# 5. Project configuration
# ---------------------------------------------------------

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]


NUM_CLASSES = len(
    CLASS_NAMES
)


IMAGE_SIZE = (
    224,
    224
)


BATCH_SIZE = 32


# ---------------------------------------------------------
# 6. Reproducibility
# ---------------------------------------------------------

SEED = 42


np.random.seed(
    SEED
)


tf.keras.utils.set_random_seed(
    SEED
)


# ---------------------------------------------------------
# 7. Environment information
# ---------------------------------------------------------

print(
    "\nTensorFlow version:",
    tf.__version__
)

print(
    "Python version:",
    sys.version.split()[0]
)

print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Number of classes:",
    NUM_CLASSES
)

print(
    "Class names:",
    CLASS_NAMES
)


# ---------------------------------------------------------
# 8. Verify Kaggle dataset structure
#
# The dataset has:
#
# dataset/
# ├── train/
# │   ├── freshapples/
# │   ├── freshbanana/
# │   └── ...
# └── test/
#     ├── freshapples/
#     ├── freshbanana/
#     └── ...
# ---------------------------------------------------------

for split_name in [
    "train",
    "test"
]:

    split_dir = (
        DATASET_ROOT
        / split_name
    )

    assert split_dir.exists(), (
        f"Missing dataset split directory: "
        f"{split_dir}"
    )

    for class_name in CLASS_NAMES:

        class_dir = (
            split_dir
            / class_name
        )

        assert class_dir.exists(), (
            f"Missing class directory: "
            f"{class_dir}"
        )


print(
    "\nTrain/test dataset structure verified."
)


# ---------------------------------------------------------
# 9. Count all dataset images
# ---------------------------------------------------------

total_images = 0


for split_name in [
    "train",
    "test"
]:

    split_total = 0

    for class_name in CLASS_NAMES:

        class_dir = (
            DATASET_ROOT
            / split_name
            / class_name
        )

        class_count = sum(
            1
            for path in class_dir.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower()
                in IMAGE_EXTENSIONS
            )
        )

        print(
            f"{split_name}/{class_name}: "
            f"{class_count}"
        )

        split_total += class_count

    print(
        f"{split_name} total: "
        f"{split_total}"
    )

    total_images += split_total


print(
    "\nTotal dataset images:",
    total_images
)


assert total_images == 13599, (
    f"Expected 13599 images, "
    f"found {total_images}"
)


# ---------------------------------------------------------
# 10. Verify repository files needed by the project
# ---------------------------------------------------------

required_repo_files = [
    REPO_DIR / "requirements.txt",
    REPO_DIR / "src" / "config.py",
    REPO_DIR / "src" / "make_time_block_split.py",
    REPO_DIR / "data" / "train.csv",
    REPO_DIR / "data" / "validation.csv",
    REPO_DIR / "data" / "test.csv",
    REPO_DIR / "data" / "session_aware" / "train_session_aware.csv",
    REPO_DIR / "data" / "session_aware" / "validation_session_aware.csv",
    REPO_DIR / "data" / "session_aware" / "test_session_aware.csv",
    REPO_DIR / "data" / "time_block" / "train_time_block.csv",
    REPO_DIR / "data" / "time_block" / "validation_time_block.csv",
    REPO_DIR / "data" / "time_block" / "test_time_block.csv",
]


for required_file in required_repo_files:

    assert required_file.exists(), (
        f"Required project file not found: "
        f"{required_file}"
    )


print(
    "\nRequired repository files verified."
)


# ---------------------------------------------------------
# 11. Final setup verification
# ---------------------------------------------------------

print(
    "\n"
    + "=" * 60
)

print(
    "RESUME / SETUP verification passed."
)

print(
    "=" * 60
)

## 1. Build Image Index

In [ ]:
records = []

for split_name in ["train", "test"]:
    split_dir = DATASET_ROOT / split_name

    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name

        for image_path in class_dir.iterdir():
            if (
                image_path.is_file()
                and image_path.suffix.lower() in IMAGE_EXTENSIONS
            ):
                records.append(
                    {
                        "original_split": split_name,
                        "class_name": class_name,
                        "image_path": str(
                            image_path.relative_to(DATASET_ROOT)
                        ),
                    }
                )

print("Total image records:", len(records))

## 2. Create the Source-Group-Aware Stratified Split

In [ ]:
import pandas as pd

df = pd.DataFrame(records)

print("DataFrame shape:", df.shape)
print("\nFirst 5 records:")
display(df.head())

In [ ]:
print("\nClass distribution:")
print(df["class_name"].value_counts())

print("\nOriginal split distribution:")
print(df["original_split"].value_counts())

In [ ]:
import re
from pathlib import Path

AUGMENTATION_PATTERNS = [
    r"^rotated_by_\d+_",
    r"^vertical_flip_",
    r"^translation_",
    r"^saltandpepper_",
]

def extract_source_filename(image_path):
    filename = Path(image_path).name

    for pattern in AUGMENTATION_PATTERNS:
        filename = re.sub(pattern, "", filename, count=1)

    return filename

df["source_filename"] = df["image_path"].apply(extract_source_filename)

df["source_id"] = df["source_filename"] + "_" + df["class_name"]

In [ ]:
print(df[["class_name", "source_filename", "source_id"]].head())
print("source_id column exists:", "source_id" in df.columns)

## 3. Verify the Leak-Free Split

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Create one row per unique source image.
source_groups = (
    df[["source_id", "class_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique source groups:", len(source_groups))

# First split: 70% training, 30% temporary.
split_1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_group_idx, temp_group_idx = next(
    split_1.split(
        source_groups["source_id"],
        source_groups["class_name"]
    )
)

train_source_ids = set(
    source_groups.iloc[train_group_idx]["source_id"]
)

temp_source_groups = source_groups.iloc[temp_group_idx].reset_index(
    drop=True
)

# Second split: 15% validation, 15% testing.
split_2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_group_idx, test_group_idx = next(
    split_2.split(
        temp_source_groups["source_id"],
        temp_source_groups["class_name"]
    )
)

validation_source_ids = set(
    temp_source_groups.iloc[val_group_idx]["source_id"]
)

test_source_ids = set(
    temp_source_groups.iloc[test_group_idx]["source_id"]
)

# Assign every image to the split of its source group.
df_leak_free = df.copy()

df_leak_free["split"] = df_leak_free["source_id"].apply(
    lambda source_id:
        "train"
        if source_id in train_source_ids
        else "validation"
        if source_id in validation_source_ids
        else "test"
)

# Create the names used by the later preprocessing cells.
train_df = df_leak_free[
    df_leak_free["split"] == "train"
].copy()

val_df = df_leak_free[
    df_leak_free["split"] == "validation"
].copy()

test_df = df_leak_free[
    df_leak_free["split"] == "test"
].copy()

print("\nImage counts:")
print("Training images  :", len(train_df))
print("Validation images:", len(val_df))
print("Testing images   :", len(test_df))
print(
    "Total images     :",
    len(train_df) + len(val_df) + len(test_df)
)

In [ ]:
print("Training distribution:")
print(train_df["class_name"].value_counts())

print("\nValidation distribution:")
print(val_df["class_name"].value_counts())

print("\nTesting distribution:")
print(test_df["class_name"].value_counts())

In [ ]:
train_sources = set(
    df_leak_free.loc[
        df_leak_free["split"] == "train",
        "source_id"
    ]
)

validation_sources = set(
    df_leak_free.loc[
        df_leak_free["split"] == "validation",
        "source_id"
    ]
)

test_sources = set(
    df_leak_free.loc[
        df_leak_free["split"] == "test",
        "source_id"
    ]
)

train_validation_overlap = train_sources & validation_sources
train_test_overlap = train_sources & test_sources
validation_test_overlap = validation_sources & test_sources

print("Train ↔ Validation overlap:", len(train_validation_overlap))
print("Train ↔ Test overlap:", len(train_test_overlap))
print("Validation ↔ Test overlap:", len(validation_test_overlap))

assert len(train_validation_overlap) == 0
assert len(train_test_overlap) == 0
assert len(validation_test_overlap) == 0

print("\nAll source-overlap checks passed.")

## 4. Export the Leak-Free Dataset Splits

In [ ]:
from pathlib import Path

LEAK_FREE_OUTPUT_DIR = Path("/content/fruit_split_metadata_leak_free")
LEAK_FREE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_COLUMNS = [
    "image_path",
    "class_name",
    "original_split",
    "source_filename",
    "source_id",
    "split",
]

train_df[CSV_COLUMNS].to_csv(
    LEAK_FREE_OUTPUT_DIR / "train.csv",
    index=False
)

val_df[CSV_COLUMNS].to_csv(
    LEAK_FREE_OUTPUT_DIR / "validation.csv",
    index=False
)

test_df[CSV_COLUMNS].to_csv(
    LEAK_FREE_OUTPUT_DIR / "test.csv",
    index=False
)

print("Leak-free CSV files created.")

print("\nRow counts:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print(
    "Total:",
    len(train_df) + len(val_df) + len(test_df)
)

In [ ]:
print("Train CSV rows      :", len(train_df))
print("Validation CSV rows :", len(val_df))
print("Test CSV rows       :", len(test_df))

## 5. Load and Verify Saved Dataset Splits

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/fruit_split_metadata_leak_free")

train_split = pd.read_csv(DATA_DIR / "train.csv")
validation_split = pd.read_csv(DATA_DIR / "validation.csv")
test_split = pd.read_csv(DATA_DIR / "test.csv")
print("Training rows  :", len(train_split))
print("Validation rows:", len(validation_split))
print("Testing rows   :", len(test_split))
print("Total rows     :", len(train_split) + len(validation_split) + len(test_split))

print("\nColumns:")
print(train_split.columns.tolist())

print("\nFirst 5 training records:")
display(train_split.head())

Training rows  : 9519
Validation rows: 2040
Testing rows   : 2040
Total rows     : 13599

Columns:
['image_path', 'class_name']

First 5 training records:


,image_path,class_name
0,train/freshbanana/vertical_flip_Screen Shot 20...,freshbanana
1,test/rottenbanana/rotated_by_30_Screen Shot 20...,rottenbanana
2,train/rottenbanana/Screen Shot 2018-06-12 at 8...,rottenbanana
3,train/rottenapples/Screen Shot 2018-06-07 at 2...,rottenapples
4,train/rottenoranges/rotated_by_15_Screen Shot ...,rottenoranges


## 6. Image Preprocessing Configuration

Images will be converted to RGB and resized to 224 × 224 pixels.

The images will be normalized before being provided to the models.

In [ ]:
IMAGE_SIZE = (224, 224)
NUM_CHANNELS = 3

print("Target image size:", IMAGE_SIZE)
print("Number of channels:", NUM_CHANNELS)

## 7. Test Image Preprocessing

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Take the first image from the training split
sample_path = train_df.iloc[0]["image_path"]

full_sample_path = DATASET_ROOT / sample_path

with Image.open(full_sample_path) as image:
    image = image.convert("RGB")
    image = image.resize(IMAGE_SIZE)

    image_array = np.array(image, dtype=np.float32)
    normalized_image = image_array / 255.0

print("Image shape:", normalized_image.shape)
print("Minimum pixel value:", normalized_image.min())
print("Maximum pixel value:", normalized_image.max())

## 8. Reusable Image Preprocessing Function

In [ ]:
from PIL import Image
import numpy as np


def preprocess_image(image_path):
    """
    Load and preprocess one image.

    Steps:
    1. Open image
    2. Convert to RGB
    3. Resize to target size
    4. Convert to float32 NumPy array
    5. Normalize pixel values to 0-1
    """
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        image = image.resize(IMAGE_SIZE)

        image_array = np.array(image, dtype=np.float32)
        normalized_image = image_array / 255.0

    return normalized_image

In [ ]:
sample_path = DATASET_ROOT / train_df.iloc[0]["image_path"]

processed_image = preprocess_image(sample_path)

print("Shape:", processed_image.shape)
print("Data type:", processed_image.dtype)
print("Minimum:", processed_image.min())
print("Maximum:", processed_image.max())

## 9. Small Batch Preprocessing Test

In [ ]:
sample_paths = train_df["image_path"].head(6)

processed_images = []

for image_path in sample_paths:
    full_path = DATASET_ROOT / image_path
    processed_image = preprocess_image(full_path)
    processed_images.append(processed_image)

processed_images = np.array(processed_images)

print("Batch shape:", processed_images.shape)
print("Data type:", processed_images.dtype)
print("Minimum:", processed_images.min())
print("Maximum:", processed_images.max())

## 10. TensorFlow Training Input Pipeline

The preprocessed images will be provided to the model using TensorFlow's `tf.data` pipeline.

Images are loaded in batches instead of loading the complete dataset into memory.

In [ ]:
import tensorflow as tf
import numpy as np

# Convert class names into numeric labels.
CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

print("Class mapping:")
for class_name, index in CLASS_TO_INDEX.items():
    print(f"{class_name} -> {index}")


def load_and_preprocess_image(image_path, label):
    """
    Load one image and prepare it for the CNN.
    """
    image_bytes = tf.io.read_file(image_path)

    # Decode RGB images directly into 3 channels.
    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    # Ensure TensorFlow knows the image has 3 channels.
    image.set_shape([None, None, 3])

    # Resize.
    image = tf.image.resize(image, IMAGE_SIZE)

    # Convert to float32 and normalize to 0-1.
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


def create_dataset(dataframe, shuffle=False):
    """
    Create a TensorFlow dataset from image paths and class labels.
    """
    image_paths = [
        str(DATASET_ROOT / image_path)
        for image_path in dataframe["image_path"]
    ]

    labels = (
        dataframe["class_name"]
        .map(CLASS_TO_INDEX)
        .to_numpy(dtype=np.int32)
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(32)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
train_ds = create_dataset(train_df, shuffle=True)
validation_ds = create_dataset(val_df, shuffle=False)
test_ds = create_dataset(test_df, shuffle=False)

print("Datasets created successfully.")

In [ ]:
for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Minimum pixel value:", tf.reduce_min(images).numpy())
    print("Maximum pixel value:", tf.reduce_max(images).numpy())

## 11. Create the Time-Block Dataset Split

The source-group split prevents exact augmented-copy leakage, but timestamp
analysis showed that neighbouring screenshots may still be very similar.

The time-block split therefore groups contiguous source photos into blocks and
keeps a minimum 60-second capture-time gap between different dataset splits.

The split is created by `src/make_time_block_split.py`.

This section creates the new time-block CSV files and does not overwrite the
existing source-group or session-aware CSV files.

In [ ]:
# ---------------------------------------------------------
# Create the new time-block dataset split
# ---------------------------------------------------------

from pathlib import Path
from src.make_time_block_split import build_time_block_split


# The notebook should be running from the repository root.
REPO_ROOT = Path.cwd()


SCRIPT_PATH = (
    REPO_ROOT
    / "src"
    / "make_time_block_split.py"
)


SESSION_AWARE_DIR = (
    REPO_ROOT
    / "data"
    / "session_aware"
)


TIME_BLOCK_DIR = (
    REPO_ROOT
    / "data"
    / "time_block"
)


# Verify that the required repository files exist.
assert SCRIPT_PATH.exists(), (
    f"Time-block script not found: {SCRIPT_PATH}"
)

assert SESSION_AWARE_DIR.exists(), (
    f"Session-aware data directory not found: {SESSION_AWARE_DIR}"
)


# Run the time-block split builder.
time_block_train_df, time_block_validation_df, time_block_test_df = (
    build_time_block_split(
        input_dir=SESSION_AWARE_DIR,
        output_dir=TIME_BLOCK_DIR,
        save=True
    )
)


# ---------------------------------------------------------
# Verify output files
# ---------------------------------------------------------

TIME_BLOCK_FILES = [
    TIME_BLOCK_DIR / "train_time_block.csv",
    TIME_BLOCK_DIR / "validation_time_block.csv",
    TIME_BLOCK_DIR / "test_time_block.csv",
]


for file_path in TIME_BLOCK_FILES:
    assert file_path.exists(), (
        f"Expected output file not found: {file_path}"
    )


print("\nTime-block split verification")
print("-" * 50)

print(
    "Training rows  :",
    len(time_block_train_df)
)

print(
    "Validation rows:",
    len(time_block_validation_df)
)

print(
    "Testing rows   :",
    len(time_block_test_df)
)

print(
    "Total rows     :",
    len(time_block_train_df)
    + len(time_block_validation_df)
    + len(time_block_test_df)
)

print(
    "\nTime-block CSV files:"
)

for file_path in TIME_BLOCK_FILES:
    print(
        file_path
    )

print(
    "\nTime-block split creation and verification completed."
)